# Example 1 — Bring Your Own NDVI Data

No Google Earth Engine required. Load any NDVI time-series CSV and estimate
the current phenological stage.

### Input requirements

Your DataFrame must have at least two columns:

| Column | Type | Description |
|--------|------|-------------|
| `date` | datetime-like | Observation date (any parseable format) |
| `NDVI` | float [0–1]   | Cloud-free NDVI value |

Observations can be irregularly spaced (e.g., every 5–12 days from satellite).
The library resamples to daily resolution internally.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from crop_stage import run_crop_stage_from_dataframe, smooth_daily_interpolate_ndvi, estimate_stage_adaptive

## 1. Load sample data

In [ ]:
df = pd.read_csv("../sample_data/sample_ndvi.csv", parse_dates=["date"])
print(f"{len(df)} observations, {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 2. One-liner: smooth + estimate

In [ ]:
result = run_crop_stage_from_dataframe(df)   # id_col=None → single field → dict
for k, v in result.items():
    print(f"  {k:20s}: {v}")

## 3. Visualise the smoothed NDVI curve

In [ ]:
df_smooth = smooth_daily_interpolate_ndvi(df)

fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(df["date"], df["NDVI"], color="gray", alpha=0.6, s=40, label="Raw NDVI", zorder=3)
ax.plot(df_smooth["date"], df_smooth["NDVI_smooth"], color="steelblue", lw=2, label="Smoothed NDVI")

ax.axhline(result["Lower_threshold"], color="orange", ls="--", label=f"Lower threshold ({result['Lower_threshold']:.2f})")
ax.axhline(result["Upper_threshold"], color="green",  ls="--", label=f"Upper threshold ({result['Upper_threshold']:.2f})")

if result.get("Peak_date"):
    ax.axvline(result["Peak_date"], color="purple", ls=":", label=f"Peak ({result['Peak_date'].strftime('%d-%b-%Y')})")

ax.set_ylim(0, 1)
ax.set_ylabel("NDVI")
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
ax.legend(loc="upper left")
ax.set_title(f"Stage: {result['Stage']} — {result['Stage_description']}")
plt.tight_layout()
plt.show()

## 4. Multiple fields at once

If your DataFrame contains multiple fields, pass `id_col` to process them all:

In [ ]:
results_df = run_crop_stage_from_dataframe(df, id_col="field_id")
results_df

## 5. Tune the thresholds (optional)

For crops with lower peak NDVI (e.g., peanuts, some cereals), you can lower the
peak detection floor:

In [ ]:
df_smooth = smooth_daily_interpolate_ndvi(df)
result_tuned = estimate_stage_adaptive(
    df_smooth["NDVI_smooth"].to_numpy(),
    dates=df_smooth["date"],
    upper_percentile=75,   # less strict peak detection
    min_peak_ndvi=0.40,    # lower floor for short-stature crops
    lower_threshold=0.30,
)
print(result_tuned["Stage"], "—", result_tuned["Stage_description"])